# Crop Type Classification with Sentinel-2 Imagery

This notebook builds a machine learning pipeline that classifies crop type for field plots in the Luís Eduardo Magalhães (LEM) region of western Bahia, Brazil. It combines the LEM+ ground-truth dataset from Embrapa with Sentinel-2 imagery.

## 1. Load the LEM+ dataset

LEM+ provides one polygon per field and one land-use label per month, from October 2019 to September 2020. See `data/README.md` for how to download it.

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

SHAPEFILE = Path("data/raw/LEM_dataset.shp")
MONTHS = [
    "Oct_2019", "Nov_2019", "Dec_2019", "Jan_2020", "Feb_2020", "Mar_2020",
    "Apr_2020", "May_2020", "Jun_2020", "Jul_2020", "Aug_2020", "Sep_2020",
]

fields = gpd.read_file(SHAPEFILE)
print(f"{len(fields)} fields, CRS: {fields.crs}")
fields.head()

1854 fields, CRS: EPSG:4326


,id,Oct_2019,Nov_2019,Dec_2019,Jan_2020,Feb_2020,Mar_2020,Apr_2020,May_2020,Jun_2020,Jul_2020,Aug_2020,Sep_2020,Note,geometry
0,0,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Conversion area,Conversion area,Conversion area,Conversion area,Conversion area,Conversion area,Conversion area,NaN,"POLYGON ((-45.82294 -12.14292, -45.8228 -12.14..."
1,1,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,NaN,"POLYGON ((-45.82389 -12.14325, -45.8176 -12.14..."
2,2,Pasture,Pasture,Pasture,Pasture,Pasture,Pasture,Pasture,Pasture,Pasture,Pasture,Pasture,Pasture,NaN,"POLYGON ((-45.82461 -12.14622, -45.81824 -12.1..."
3,3,Coffee,Coffee,Coffee,Coffee,Coffee,Coffee,Coffee,Coffee,Coffee,Coffee,Coffee,Coffee,NaN,"POLYGON ((-45.81202 -12.14691, -45.81194 -12.1..."
4,4,Not identified,Uncultivated soil,Uncultivated soil,Uncultivated soil,Soybean,Soybean,Soybean,Uncultivated soil,Uncultivated soil,Uncultivated soil,Uncultivated soil,Uncultivated soil,NaN,"POLYGON ((-45.83022 -12.15034, -45.83024 -12.1..."


### Validate the polygons

The rest of the pipeline depends on one valid polygon per field, so these checks stop the notebook if that does not hold. Areas are computed in a projected UTM CRS, because degrees cannot be used to measure area.

In [2]:
assert fields["id"].is_unique, "Field ids must be unique"
assert fields[MONTHS].notna().all().all(), "Every field needs a label for every month"

print("Missing geometries:", int(fields.geometry.isna().sum()))
print("Invalid geometries:", int((~fields.geometry.is_valid).sum()))

utm_crs = fields.estimate_utm_crs()
fields["area_ha"] = fields.to_crs(utm_crs).area / 10_000
print("Projected CRS:", utm_crs.name)
fields["area_ha"].describe().round(1)

Missing geometries: 0
Invalid geometries: 0


Projected CRS: WGS 84 / UTM zone 23S


count    1854.0
mean      135.3
std       135.8
min         0.1
25%        50.6
50%       101.7
75%       181.7
max      2620.5
Name: area_ha, dtype: float64

### Reshape the labels

The shapefile stores one column per month. A long table, with one row per field and month, is the format needed to join the labels with the Sentinel-2 time series later.

In [3]:
labels = fields.melt(
    id_vars="id", value_vars=MONTHS, var_name="month", value_name="label"
)
labels["month"] = pd.to_datetime(labels["month"], format="%b_%Y")

print(f"{len(labels)} field-month rows, {labels['label'].nunique()} classes")
labels.head()

22248 field-month rows, 16 classes


,id,month,label
0,0,2019-10-01,Cerrado
1,1,2019-10-01,Cerrado
2,2,2019-10-01,Pasture
3,3,2019-10-01,Coffee
4,4,2019-10-01,Not identified


In [4]:
class_by_month = (
    labels.pivot_table(index="label", columns="month", aggfunc="size", fill_value=0)
    .rename(columns=lambda month: month.strftime("%b %Y"))
)
class_by_month.loc[class_by_month.sum(axis=1).sort_values(ascending=False).index]

month,Oct 2019,Nov 2019,Dec 2019,Jan 2020,Feb 2020,Mar 2020,Apr 2020,May 2020,Jun 2020,Jul 2020,Aug 2020,Sep 2020
label,,,,,,,,,,,,
Uncultivated soil,1433,1463,1307,162,38,477,1061,949,420,480,669,1137
Soybean,0,4,126,1133,1173,712,111,4,1,1,0,0
Cerrado,161,161,159,158,157,151,149,147,144,139,138,137
Millet,0,4,5,4,5,5,8,106,406,422,332,64
Pasture,95,95,95,95,95,97,97,102,102,102,102,102
Corn,0,0,66,158,168,175,181,184,131,56,36,13
Brachiaria,6,6,7,7,7,7,11,53,207,249,250,222
Cotton,0,0,1,48,121,138,138,138,137,118,71,16
Sorghum,0,0,0,1,3,3,7,51,161,163,147,60


### How often does a field change class?

A field can hold several land uses within one season, for example soybean in summer followed by millet in winter. This matters when defining the prediction target, because a single label per season has to be derived from the monthly labels.

In [5]:
classes_per_field = fields[MONTHS].nunique(axis=1)
classes_per_field.value_counts().sort_index().rename_axis("classes in season").rename("fields")

classes in season
1    313
2    559
3    929
4     53
Name: fields, dtype: int64

## 2. Extract Sentinel-2 composites with Google Earth Engine

The labels are monthly, so the imagery is aggregated to the same resolution: one composite per month, from October 2019 to September 2020.

For each month, the pipeline:

1. Selects Sentinel-2 L2A surface reflectance images over the study area
2. Masks clouds, cloud shadows and cirrus with the Scene Classification Layer (SCL)
3. Takes the per-pixel median of the clear observations
4. Computes NDVI, EVI and NDWI from the composite
5. Averages the pixels inside each field polygon

The resulting table is stored in `data/sentinel2_features.csv`, so the rest of the notebook runs without an Earth Engine account. To rebuild it, delete the CSV, replace `EE_PROJECT` with your own Cloud project and run `earthengine authenticate` once.

In [6]:
import ee

EE_PROJECT = "crop-classification-sentinel2"
FEATURES_CSV = Path("data/sentinel2_features.csv")

COLLECTION = "COPERNICUS/S2_SR_HARMONIZED"
BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]
INDICES = ["NDVI", "EVI", "NDWI"]
CLEAR_SCL_CLASSES = [4, 5, 6, 7]  # vegetation, bare soil, water, unclassified
SCALE_METERS = 20  # native resolution of the SWIR bands
CHUNK_SIZE = 400  # fields per request, to stay within Earth Engine limits

needs_extraction = not FEATURES_CSV.exists()
if needs_extraction:
    ee.Initialize(project=EE_PROJECT)
    study_area = ee.Geometry.Rectangle([float(value) for value in fields.total_bounds])

In [7]:
def mask_clouds(image):
    """Keep clear pixels and convert the bands to surface reflectance."""
    scl = image.select("SCL")
    clear = ee.Image(0)
    for scl_class in CLEAR_SCL_CLASSES:
        clear = clear.Or(scl.eq(scl_class))
    return image.select(BANDS).divide(10_000).updateMask(clear)


def monthly_composite(month_start):
    """Build a cloud-free median composite with vegetation indices for one month."""
    start = ee.Date(month_start)
    images = (
        ee.ImageCollection(COLLECTION)
        .filterBounds(study_area)
        .filterDate(start, start.advance(1, "month"))
        .map(mask_clouds)
    )
    composite = images.median()
    ndvi = composite.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndwi = composite.normalizedDifference(["B3", "B8"]).rename("NDWI")
    evi = composite.expression(
        "2.5 * (NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1)",
        {
            "NIR": composite.select("B8"),
            "RED": composite.select("B4"),
            "BLUE": composite.select("B2"),
        },
    ).rename("EVI")
    # clear_obs counts how many cloud-free images contributed to each pixel
    clear_obs = images.select("B4").count().rename("clear_obs")
    return composite.addBands([ndvi, evi, ndwi, clear_obs])


def extract_month(month_start, polygons):
    """Return the mean band and index values per field for one month."""
    composite = monthly_composite(month_start)
    rows = []
    for offset in range(0, len(polygons), CHUNK_SIZE):
        chunk = polygons.iloc[offset : offset + CHUNK_SIZE][["id", "geometry"]]
        result = composite.reduceRegions(
            collection=ee.FeatureCollection(chunk.__geo_interface__),
            reducer=ee.Reducer.mean(),
            scale=SCALE_METERS,
            tileScale=4,
        ).getInfo()
        rows += [feature["properties"] for feature in result["features"]]
    table = pd.DataFrame(rows)
    table.insert(1, "month", pd.Timestamp(month_start))
    return table

### Build the feature table

This produces one row per field and month, with the band and index values plus that month's label. The extraction runs month by month and is skipped when the CSV already exists.

In [8]:
FEATURE_COLUMNS = BANDS + INDICES + ["clear_obs"]

if needs_extraction:
    months = pd.date_range("2019-10-01", periods=len(MONTHS), freq="MS")
    monthly_tables = []
    for month in months:
        monthly_tables.append(extract_month(month.strftime("%Y-%m-%d"), fields))
        print(f"{month:%b %Y}: {len(monthly_tables[-1])} fields")

    extracted = pd.concat(monthly_tables, ignore_index=True)
    extracted[FEATURE_COLUMNS] = extracted[FEATURE_COLUMNS].round(5)
    features = extracted.merge(labels, on=["id", "month"], how="left", validate="one_to_one")
    features = features[["id", "month", "label"] + FEATURE_COLUMNS]
    features.sort_values(["id", "month"]).to_csv(FEATURES_CSV, index=False)

features = pd.read_csv(FEATURES_CSV, dtype={"id": str}, parse_dates=["month"])
print(f"{len(features)} rows, {features['id'].nunique()} fields, {features['month'].nunique()} months")
features.head()

Oct 2019: 1854 fields


Nov 2019: 1854 fields


Dec 2019: 1854 fields


Jan 2020: 1854 fields


Feb 2020: 1854 fields


Mar 2020: 1854 fields


... (4 outputs left out)

Aug 2020: 1854 fields


Sep 2020: 1854 fields


22248 rows, 1854 fields, 12 months


,id,month,label,B2,B3,B4,B8,B11,B12,NDVI,EVI,NDWI,clear_obs
0,0,2019-10-01,Cerrado,0.06072,0.08878,0.11190,0.25635,0.32264,0.21751,0.39272,0.24535,-0.48616,5.00000
1,0,2019-11-01,Cerrado,0.05885,0.08958,0.09870,0.25522,0.31051,0.20294,0.44289,0.27854,-0.48072,2.00000
2,0,2019-12-01,Cerrado,0.04779,0.07765,0.06973,0.26892,0.26428,0.16247,0.58870,0.37522,-0.55180,3.23512
3,0,2020-01-01,Cerrado,0.06890,0.09175,0.07799,0.29829,0.25986,0.15095,0.58649,0.44027,-0.52957,2.96361
4,0,2020-02-01,Cerrado,0.12118,0.13713,0.12480,0.29330,0.23407,0.15318,0.42773,0.37341,-0.38002,1.33234


### Check the data quality

Every field and month should be present, with physically plausible values. Rainy months are the main risk, because a field with no clear observation has no values at all. `clear_obs` shows how many cloud-free images support each monthly value.

In [9]:
assert len(features) == len(fields) * len(MONTHS), "Expected one row per field and month"
assert features["label"].notna().all(), "Every row needs a label"
assert features[["NDVI", "NDWI"]].stack().between(-1, 1).all(), "Normalized indices must be between -1 and 1"
assert features[BANDS].stack().ge(0).all(), "Reflectance cannot be negative"

quality = features.groupby("month").agg(
    missing_ndvi=("NDVI", lambda values: int(values.isna().sum())),
    median_clear_obs=("clear_obs", "median"),
    min_clear_obs=("clear_obs", "min"),
    median_ndvi=("NDVI", "median"),
).round(2)
quality.index = quality.index.strftime("%b %Y")
quality

,missing_ndvi,median_clear_obs,min_clear_obs,median_ndvi
month,,,,
Oct 2019,0,5.00,2.95,0.17
Nov 2019,0,3.04,1.00,0.20
Dec 2019,0,3.74,1.79,0.26
Jan 2020,0,3.16,1.25,0.76
Feb 2020,0,2.42,1.00,0.77
Mar 2020,0,2.00,1.00,0.50
Apr 2020,0,2.48,1.00,0.27
May 2020,0,3.00,1.01,0.45
Jun 2020,0,5.55,3.07,0.52
